# IMH-LVS label-extraction fine-tune — Colab T4 (7B)

Trains a bigger label-reading model (Qwen2-VL-**7B**, vs. the 2B this project runs on the local 6GB laptop GPU) using Colab's free T4 GPU (16GB VRAM).

**Before you start:** Runtime menu -> Change runtime type -> T4 GPU.

**What you need on hand:** `all_labels.zip` (the zipped `all labels` folder — already prepared for you locally, sitting next to this notebook's project root).

**Data handling:** nothing here uses Google Drive to *store* your label data — the zip is uploaded directly into this temporary Colab session and disappears when the session ends. Drive is used only, optionally, to save training checkpoints as a safety net against disconnects (see the toggle below) — that's the trained result, not your source label images.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Install dependencies

Poppler (for reading PDFs) plus the Python packages the fine-tuning scripts need. Torch is deliberately **not** reinstalled — Colab's pre-installed torch is already matched to its CUDA driver; installing a different one is a common way to break GPU access.

In [ ]:
!apt-get -qq update && apt-get -qq install -y poppler-utils
!pip install -q "transformers>=4.46.0" accelerate peft bitsandbytes pdf2image pillow pydantic

## 2. Get the code

Clones the project's public GitHub repo — this brings the fine-tuning scripts AND the already-reviewed ground-truth annotations (`ai_backend/finetune/reviewed/annotations/`), which are committed. It does **not** bring the raw label files (those are deliberately not committed) — that's the zip you'll upload next.

In [ ]:
!git clone -q https://github.com/Sayan59das/project.git
%cd project/IMH-LVS/ai_backend

## 3. Upload your label files

Run this cell, then click "Choose Files" and pick `all_labels.zip` from your computer. It goes straight into this session's temporary storage — not Drive.

In [ ]:
from google.colab import files
uploaded = files.upload()  # pick all_labels.zip

import zipfile
zip_name = next(iter(uploaded))
with zipfile.ZipFile(zip_name) as zf:
    zf.extractall("../")  # unzips to IMH-LVS/all labels/, matching prepare_dataset.py's SOURCE_DIRS
print("Uploaded and extracted:", zip_name)

## 4. Build the training set

Same two scripts the local pipeline uses, run fresh here so the dataset's file paths point at THIS machine, not your laptop's. Rasterizes every raw label into page images, then pairs the ones with reviewed annotations into `dataset/train.jsonl` + `dataset/val.jsonl` (grouped by real product family, not filename — see `build_training_set.py` for why).

In [ ]:
!PYTHONPATH=. python finetune/prepare_dataset.py

In [ ]:
!PYTHONPATH=. python finetune/build_training_set.py

## 5. Optional: checkpoint to Drive

Colab can disconnect you mid-training. `train_lora.py` already saves the adapter every time it beats its best validation loss so far — pointing that save location at a Drive folder means a disconnect only loses progress since the last improved epoch, not the whole run.

**This only affects where the small trained result gets saved — your label images are never touched by this step.** Set `USE_DRIVE_CHECKPOINT = False` below to skip Drive entirely and keep everything in this temporary session (you'll download the result manually in step 7 instead).

In [ ]:
USE_DRIVE_CHECKPOINT = True

import os
os.environ["FINETUNE_MODEL_PATH"] = "Qwen/Qwen2-VL-7B-Instruct"

if USE_DRIVE_CHECKPOINT:
    from google.colab import drive
    drive.mount("/content/drive")
    os.environ["FINETUNE_ADAPTER_OUT"] = "/content/drive/MyDrive/imh-lvs-label-extraction-lora-7b"
    print("Adapter will be saved to Drive at:", os.environ["FINETUNE_ADAPTER_OUT"])
else:
    print("Adapter will stay in this temporary session — download it in step 7 before the session ends.")

## 6. Train

Unchanged training loop from the local script — only the model size and save location differ, via the env vars set above. 16GB of VRAM here vs. 6GB locally, so this comfortably fits where the 2B was already tight.

In [ ]:
!PYTHONPATH=. python finetune/train_lora.py

## 7. Get the trained adapter back

If you used Drive checkpointing (step 5), the result is already in your Drive at the path printed above — nothing more to do. Otherwise, run this to download it directly to your computer.

In [ ]:
import os
if not USE_DRIVE_CHECKPOINT:
    adapter_dir = os.environ.get("FINETUNE_ADAPTER_OUT", "adapters/label-extraction-lora")
    !zip -qr adapter_7b.zip "{adapter_dir}"
    from google.colab import files
    files.download("adapter_7b.zip")
else:
    print("Already saved to Drive — nothing to download here.")